# D2.6 · Post-incident change surface

**Function D — Security Operations → The Incident Responder**  ·  *Security of AI*

---

**Risk.** Fixing the prompt when the bug is in the control plane.

**Control.** Choose among model, prompt, tool, policy, sandbox, identity, eval.

**This lab.** Pick the right layer to change after an incident.

| | |
|---|---|
| Open-source tooling | — |
| Open-weight models | — |

> Runs anywhere: standard library only, no network, no API key. Where a lesson names a real tool you would deploy (Falco, OPA, SPIRE, Keycloak), the notebook models the *decision* that tool makes, so the lesson still lands on a machine that cannot pull containers.

In [ ]:
# --- Cyber Commons bootstrap -------------------------------------------------
# Puts the lab library on the path. Works from a clone, from the repo root, and
# on Kaggle. Standard library only — nothing to install, no network required.
import sys, os, subprocess
from pathlib import Path

def _find_labs():
    for base in [Path.cwd(), *Path.cwd().parents]:
        if (base / "labs" / "cybercommons" / "__init__.py").is_file():
            return base / "labs"
    # Kaggle kernels start in /kaggle/working with the repo absent. If the
    # kernel has internet enabled we clone it; if not, this raises and the
    # message tells you to attach the repo as a dataset instead.
    dest = Path("/kaggle/working/cyber-commons")
    if not dest.exists():
        subprocess.run(["git", "clone", "--depth", "1", "--branch", "claude/vulnbench-setup-scheduling-81aqov",
                        "https://github.com/spbreed/cyber-commons", str(dest)], check=True)
    return dest / "labs"

sys.path.insert(0, str(_find_labs()))
import cybercommons
print(cybercommons.banner("D2.6"))

The post-incident change surface for an agentic incident is wider than for a software one, because the fix may be a prompt, a manifest, a model version or a policy — and only one of those goes through change management.

In [ ]:
SURFACES = {
 "application code":  ("yes", "PR, review, CI, deploy"),
 "agent prompt":      ("usually not", "edited in a console, no review"),
 "tool manifest":     ("usually not", "config change, no threat-model diff"),
 "model version":     ("no", "provider-side, may change without notice"),
 "policy (OPA/rego)": ("sometimes", "depends whether it is in git"),
 "approval settings": ("rarely", "a toggle in an admin UI"),
}
print(f"{'change surface':20s}{'in change mgmt?':18s}what happens today")
for k, (managed, how) in SURFACES.items():
    print(f"{k:20s}{managed:18s}{how}")

Four of six bypass the process that exists. The A1.1 manifest diff is the cheapest way to bring two of them back in.

In [ ]:
from cybercommons import planes
W = planes.Tool
before = planes.Manifest("agent", [W("read_file")], rung="L2")
after  = planes.Manifest("agent", [W("read_file"),
                                   W("run_shell", writes=True, scope="tenant",
                                     reversible=False)], rung="L2")
d = planes.diff_manifests(before, after)
print("post-incident manifest change:", d["added"],
      f"blast {d['blast_before']} → {d['blast_after']}")
for p in d["new_problems"]:
    print("  ⚠", p)

### Expect

The table shows four of six surfaces outside change management, and the manifest diff flags the newly added irreversible ungated tool.

### Your turn

Pick the one unmanaged surface that would have prevented your last incident. Getting it into git is usually a day of work and it is the highest-leverage day available.

---

[All lessons](https://github.com/spbreed/cyber-commons/tree/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks) · [Lesson page](https://spbreed.github.io/cyber-commons/lessons/D2.6.html) · [Lab library](https://github.com/spbreed/cyber-commons/tree/claude/vulnbench-setup-scheduling-81aqov/labs/cybercommons)

*Cyber Commons — a free, open commons for Cyber AI.*